# 御厨·臻享 LoRA 微调 — Colab 一键运行

**运行环境要求**：Google Colab（T4 GPU 免费版即可）

**完成后需下载**：
- `data/lora_adapter/` — LoRA 权重文件
- `data/lora_eval_report.json` — 评估指标
- `data/lora_dataset/` — 生成的训练数据集（备用）

---
**运行顺序**：Step 0 → Step 1 → Step 2 → Step 3 → Step 4（下载）

## Step 0：环境准备

In [ ]:
# 确认 GPU 可用
import torch
print(f"GPU 可用: {torch.cuda.is_available()}")
print(f"GPU 型号: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '无'}")

In [ ]:
# 安装依赖
!pip install -q transformers>=4.40.0 peft>=0.10.0 trl>=0.8.0 datasets>=2.18.0 accelerate>=0.28.0 bitsandbytes>=0.43.0 openai>=1.30.0 tqdm

In [ ]:
import os

# 上传项目脚本（运行后点击「选择文件」上传以下三个文件）
# - dataset_builder.py
# - train_lora.py
# - evaluate.py
from google.colab import files
files.upload()

In [ ]:
# 配置 API Key（用于 Self-Instruct 数据扩充 & LLM-as-Judge 评估）
import os
from google.colab import userdata

# 方式一（推荐）：在 Colab 左侧「Secrets」中添加 DASHSCOPE_API_KEY
try:
    os.environ["DASHSCOPE_API_KEY"] = userdata.get("DASHSCOPE_API_KEY")
    print("✅ API Key 已从 Secrets 加载")
except Exception:
    # 方式二：直接填写（不推荐提交到 Git）
    os.environ["DASHSCOPE_API_KEY"] = "your_dashscope_api_key_here"
    print("⚠️  请替换为真实的 DASHSCOPE_API_KEY")

# 创建数据目录
os.makedirs("data/lora_dataset", exist_ok=True)
os.makedirs("data/lora_adapter", exist_ok=True)
print("✅ 目录结构已创建")

## Step 1：构建数据集

- 加载 100 条内置种子数据
- 调用 Qwen API 进行 Self-Instruct 扩充至 ~1000 条
- 划分 train / val / test 并保存为 JSONL

预计耗时：**5~10 分钟**（取决于 API 响应速度）

In [ ]:
!python dataset_builder.py

In [ ]:
# 验证数据集
import json
for split in ["train", "val", "test"]:
    path = f"data/lora_dataset/{split}.jsonl"
    with open(path) as f:
        count = sum(1 for _ in f)
    print(f"{split:5s}: {count} 条")

## Step 2：QLoRA 微调训练

- 基础模型：Qwen2.5-7B-Instruct
- QLoRA：4-bit NF4 量化 + LoRA（r=8, alpha=16, target q/k/v/o_proj）
- 训练轮次：3 epochs

预计耗时：**60~90 分钟**（Colab T4）

In [ ]:
!python train_lora.py

In [ ]:
import os
import json

# 验证 adapter 已保存
adapter_files = os.listdir("data/lora_adapter")
print("✅ 成功保存的 Adapter 文件:", adapter_files)

# 读取训练摘要
with open("data/lora_adapter/train_summary.json") as f:
    summary = json.load(f)

# 使用正确的 Key 来读取数据
elapsed = summary['elapsed_seconds']
minutes, seconds = divmod(int(elapsed), 60)

print("\n📊 训练成果大公开：")
print(f"⏱️ 训练耗时: {minutes} 分 {seconds} 秒 ({elapsed} 秒)")
print(f"📉 最终 Loss: {summary['final_train_loss']}")

## Step 3：评估 — Base vs Fine-tuned

对比 base Qwen2.5-7B 与 QLoRA 微调后模型在 100 条测试集上的表现。

预计耗时：**15~20 分钟**

In [ ]:
!python evaluate.py

In [3]:
# 打印核心指标（用于简历）
with open("data/lora_eval_report.json") as f:
    report = json.load(f)

base = report.get("base_model") or {}
ft   = report["finetuned_model"]
judge = report.get("judge_model", "qwen-max-latest")

def fmt(val):
    return f"{val:.1f}%" if val is not None else "N/A"

def delta(b, f):
    if b is None or f is None: return "N/A"
    d = f - b
    sign = "↑" if d >= 0 else "↓"
    return f"{sign}{abs(d):.1f} 个百分点"

b_chef = base.get("chef_accuracy_pct")
f_chef = ft.get("chef_accuracy_pct")
b_safe = base.get("food_safety_accuracy_pct")
f_safe = ft.get("food_safety_accuracy_pct")

print("=" * 58)
print("         QLoRA 微调效果 — 简历数字速查")
print("=" * 58)
print(f"裁判模型        : {judge}")
print(f"评估样本数      : {ft['sample_count']} 条")
print()
print(f"私厨建议准确率    : {fmt(b_chef)} → {fmt(f_chef)}   ({delta(b_chef, f_chef)})")
print(f"食品安全预警准确率 : {fmt(b_safe)} → {fmt(f_safe)}   ({delta(b_safe, f_safe)})")
print()
print("【简历表述参考】")
if f_chef is not None and b_chef is not None and f_safe is not None and b_safe is not None:
    d1 = f_chef - b_chef
    d2 = f_safe - b_safe
    print(f"基于 Self-Instruct 构建 1000 条私厨领域指令数据集，")
    print(f"对 Qwen2.5-7B-Instruct 进行 QLoRA 微调（4-bit NF4, r=8）；")
    if d1 >= 0:
        print(f"以 {judge} 为裁判，私厨建议准确率从 {b_chef:.0f}% 提升至 {f_chef:.0f}%（+{d1:.1f} 个百分点）；")
    else:
        print(f"以 {judge} 为裁判，私厨建议准确率 {b_chef:.0f}% → {f_chef:.0f}%；")
    if d2 >= 0:
        print(f"食品安全预警准确率从 {b_safe:.0f}% 提升至 {f_safe:.0f}%（+{d2:.1f} 个百分点）。")
    else:
        print(f"食品安全预警准确率 {b_safe:.0f}% → {f_safe:.0f}%。")

         QLoRA 微调效果 — 简历数字速查
裁判模型        : qwen-max-latest
评估样本数      : 100 条

私厨建议准确率    : 88.0% → 95.0%   (↑7.0 个百分点)
食品安全预警准确率 : 88.0% → 92.0%   (↑4.0 个百分点)

【简历表述参考】
基于 Self-Instruct 构建 1000 条私厨领域指令数据集，
对 Qwen2.5-7B-Instruct 进行 QLoRA 微调（4-bit NF4, r=8）；
以 qwen-max-latest 为裁判，私厨建议准确率从 88% 提升至 95%（+7.0 个百分点）；
食品安全预警准确率从 88% 提升至 92%（+4.0 个百分点）。


## Step 4：打包下载

将所有产物打包为 zip 下载到本地，解压后放入项目 `data/` 目录即可。

In [ ]:
import shutil
from google.colab import files

# 打包
shutil.make_archive("lora_outputs", "zip", ".", "data")
print("✅ 已打包为 lora_outputs.zip")

# 下载
files.download("lora_outputs.zip")
print("✅ 下载完成，解压后将 data/ 目录合并到本地项目即可")